In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
base_path = "/content/drive/MyDrive/LLMwithreasoner"
os.makedirs(base_path, exist_ok=True)

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets \
             sentence-transformers scikit-learn rouge-score \
             bert-score scipy tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.3 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/LLMwithreasoner/checkpoints", exist_ok=True)
os.makedirs("/content/drive/MyDrive/LLMwithreasoner/results", exist_ok=True)
os.makedirs("/content/drive/MyDrive/LLMwithreasoner/logs", exist_ok=True)
print("Folders created!")

Folders created!


In [ ]:
%%writefile /content/drive/MyDrive/LLMwithreasoner/config.py
import os
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_DIR        = os.path.dirname(os.path.abspath(__file__))
CHECKPOINT_DIR  = os.path.join(BASE_DIR, "checkpoints")
RESULTS_DIR     = os.path.join(BASE_DIR, "results")
LOG_DIR         = os.path.join(BASE_DIR, "logs")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)
os.makedirs(LOG_DIR,        exist_ok=True)

LLAMA_MODEL_ID        = "meta-llama/Meta-Llama-3-8B-Instruct"
LLAMA_MAX_NEW_TOKENS  = 256
LLAMA_COT_MAX_TOKENS  = 512
LLAMA_TEMPERATURE     = 0.7
LLAMA_GREEDY_TEMP     = 0.0
LLAMA_N_SAMPLES       = 5
LLAMA_LOAD_IN_8BIT    = True

VOCAB_SIZE            = 30_522
MAX_SEQ_LEN           = 256
D_MODEL               = 128
N_HEADS               = 4
N_LAYERS              = 3
D_FF                  = 256
DROPOUT               = 0.1
NUM_CLASSES           = 2

BATCH_SIZE            = 32
LEARNING_RATE         = 3e-4
NUM_EPOCHS            = 10
WEIGHT_DECAY          = 1e-2
WARMUP_STEPS          = 200
TRAIN_SPLIT           = 0.8
RANDOM_SEED           = 42
CLASSIFIER_CKPT       = os.path.join(CHECKPOINT_DIR, "scratch_transformer_best.pt")

ENTROPY_THRESHOLD     = 0.6
SBERT_MODEL_ID        = "all-MiniLM-L6-v2"

HALUEVAL_SPLIT        = "data"
TRUTHFULQA_SPLIT      = "validation"

print(f"[Config] Device       : {DEVICE}")
print(f"[Config] Llama Model  : {LLAMA_MODEL_ID}")
print(f"[Config] Classifier   : Scratch Transformer  (d={D_MODEL}, heads={N_HEADS}, layers={N_LAYERS})")
print(f"[Config] Entropy Thr. : {ENTROPY_THRESHOLD}")

Writing /content/drive/MyDrive/LLMwithreasoner/config.py


In [ ]:
%%writefile /content/drive/MyDrive/LLMwithreasoner/data_pipeline.py
import random
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
import config


def get_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    return tokenizer


class HaluEvalDataset(Dataset):

    def __init__(self, tokenizer, split="train", max_len=config.MAX_SEQ_LEN):
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.samples   = []

        print(f"\n[HaluEval] Loading dataset (split={split}) ...")
        raw = load_dataset("pminervini/HaluEval", "qa_samples", split="data")

        print(f"[HaluEval] Raw samples: {len(raw)}")
        self._process(raw)
        print(f"[HaluEval] Processed  : {len(self.samples)} samples  "
              f"(factual={sum(1 for s in self.samples if s['label']==0)}, "
              f"hallucination={sum(1 for s in self.samples if s['label']==1)})")

    def _process(self, raw):
        for row in tqdm(raw, desc="Tokenising HaluEval"):
            question  = str(row.get("question", "")).strip()
            right_ans = str(row.get("right_answer", "")).strip()
            hal_ans   = str(row.get("hallucinated_answer", "")).strip()

            if question and right_ans:
                enc = self._encode(question, right_ans)
                enc["label"] = 0
                self.samples.append(enc)

            if question and hal_ans:
                enc = self._encode(question, hal_ans)
                enc["label"] = 1
                self.samples.append(enc)

    def _encode(self, question, answer):
        encoded = self.tokenizer(
            question,
            answer,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
        }

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return (
            s["input_ids"],
            s["attention_mask"],
            torch.tensor(s["label"], dtype=torch.long),
        )


def get_halueval_loaders(tokenizer, batch_size=config.BATCH_SIZE,
                          train_split=config.TRAIN_SPLIT,
                          seed=config.RANDOM_SEED):
    full_dataset = HaluEvalDataset(tokenizer)
    n_train = int(len(full_dataset) * train_split)
    n_val   = len(full_dataset) - n_train

    generator = torch.Generator().manual_seed(seed)
    train_set, val_set = random_split(full_dataset, [n_train, n_val], generator=generator)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    print(f"\n[DataLoader] Train batches : {len(train_loader)}  |  Val batches : {len(val_loader)}")
    return train_loader, val_loader


def load_truthfulqa(split="validation", max_samples=None):
    print(f"\n[TruthfulQA] Loading (split={split}) ...")
    raw = load_dataset("truthful_qa", "generation", split=split)

    samples = []
    for row in raw:
        samples.append({
            "question":        row["question"],
            "best_answer":     row["best_answer"],
            "correct_answers": row["correct_answers"],
        })

    if max_samples:
        random.seed(config.RANDOM_SEED)
        samples = random.sample(samples, min(max_samples, len(samples)))

    print(f"[TruthfulQA] Loaded {len(samples)} questions")
    return samples


if __name__ == "__main__":
    tok = get_tokenizer()

    train_loader, val_loader = get_halueval_loaders(tok, batch_size=4)
    ids, mask, labels = next(iter(train_loader))
    print(f"\nSample batch — input_ids: {ids.shape}  labels: {labels}")

    tqa = load_truthfulqa(max_samples=3)
    for q in tqa:
        print(f"\nQ: {q['question']}\nA: {q['best_answer']}")


Writing /content/drive/MyDrive/LLMwithreasoner/data_pipeline.py


In [ ]:
%%writefile /content/drive/MyDrive/LLMwithreasoner/scratch_transformer.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import config


class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=config.MAX_SEQ_LEN, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class MultiHeadSelfAttention(nn.Module):

    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(p=dropout)
        self.scale = math.sqrt(self.d_k)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(context)


class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerEncoderLayer(nn.Module):

    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ff(self.norm2(x))
        return x


class ScratchTransformerClassifier(nn.Module):

    def __init__(
        self,
        vocab_size=config.VOCAB_SIZE,
        d_model=config.D_MODEL,
        n_heads=config.N_HEADS,
        n_layers=config.N_LAYERS,
        d_ff=config.D_FF,
        max_len=config.MAX_SEQ_LEN,
        dropout=config.DROPOUT,
        num_classes=config.NUM_CLASSES,
    ):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask):
        x = self.token_embedding(input_ids)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, mask=attention_mask)
        x = self.norm(x)
        cls_repr = x[:, 0, :]
        logits = self.classifier(cls_repr)
        return logits

    def predict(self, input_ids, attention_mask):
        with torch.no_grad():
            logits = self.forward(input_ids, attention_mask)
            return torch.argmax(logits, dim=-1)

    def predict_proba(self, input_ids, attention_mask):
        with torch.no_grad():
            logits = self.forward(input_ids, attention_mask)
            return F.softmax(logits, dim=-1)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


if __name__ == "__main__":
    import random
    from datasets import load_dataset
    from transformers import AutoTokenizer
    random.seed(42)

    model = ScratchTransformerClassifier().to(config.DEVICE)
    print(f"\nScratch Transformer — Trainable parameters: {model.count_parameters():,}")

    print("\n[Smoke Test] Loading HaluEval samples ...")
    halueval = load_dataset("pminervini/HaluEval", "qa_samples", split="data")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    rows = random.sample(range(len(halueval)), 4)

    questions, answers, true_labels = [], [], []
    for i, idx in enumerate(rows):
        row = halueval[idx]
        if i % 2 == 0:
            answers.append(row["right_answer"])
            true_labels.append(0)
        else:
            answers.append(row["hallucinated_answer"])
            true_labels.append(1)
        questions.append(row["question"])

    encoding = tokenizer(
        questions,
        answers,
        max_length=config.MAX_SEQ_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    input_ids = encoding["input_ids"].to(config.DEVICE)
    attention_mask = encoding["attention_mask"].to(config.DEVICE)

    logits = model(input_ids, attention_mask)
    preds = model.predict(input_ids, attention_mask)
    probs = model.predict_proba(input_ids, attention_mask)

    print(f"\n{'─'*65}")
    print(f"  {'Question':<35} {'True':>5}  {'Pred':>5}  {'P(hal)':>7}")
    print(f"{'─'*65}")
    for i in range(4):
        q_short = questions[i][:33] + ".."
        true_lbl = "HAL" if true_labels[i] == 1 else "FACT"
        pred_lbl = "HAL" if preds[i].item() == 1 else "FACT"
        p_hal = probs[i, 1].item()
        print(f"  {q_short:<35} {true_lbl:>5}  {pred_lbl:>5}  {p_hal:>7.3f}")
    print(f"{'─'*65}")
    print(f"\nNote: Predictions are random (untrained). Run train first.")

Writing /content/drive/MyDrive/LLMwithreasoner/scratch_transformer.py


In [ ]:
%%writefile /content/drive/MyDrive/LLMwithreasoner/semantic_entropy.py
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as shannon_entropy
from sentence_transformers import SentenceTransformer
import config


class SemanticEntropyScorer:

    def __init__(self, model_name=config.SBERT_MODEL_ID, threshold=config.ENTROPY_THRESHOLD):
        print(f"[SemanticEntropy] Loading sentence encoder: {model_name} ...")
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold
        print(f"[SemanticEntropy] Ready  |  threshold={threshold}")

    def compute(self, answers):
        n = len(answers)
        if n < 2:
            return 0.0

        embeddings = self.model.encode(answers, convert_to_numpy=True, show_progress_bar=False)
        sim_matrix = cosine_similarity(embeddings)
        sim_matrix = np.clip(sim_matrix, 0.0, 1.0)
        np.fill_diagonal(sim_matrix, 0.0)
        avg_sims = sim_matrix.sum(axis=1) / (n - 1)
        diversity = 1.0 - avg_sims
        diversity = np.clip(diversity, 1e-9, None)
        probs = diversity / diversity.sum()
        raw_entropy = shannon_entropy(probs)
        max_entropy = np.log(n)
        normalised = raw_entropy / max_entropy if max_entropy > 0 else 0.0
        return float(np.clip(normalised, 0.0, 1.0))

    def is_uncertain(self, answers):
        score = self.compute(answers)
        return (score > self.threshold), score

    def describe(self, answers):
        n = len(answers)
        if n < 2:
            return {"score": 0.0, "uncertain": False, "n_answers": n,
                    "avg_pairwise_sim": None, "verdict": "insufficient samples"}

        embeddings = self.model.encode(answers, convert_to_numpy=True, show_progress_bar=False)
        sim_matrix = cosine_similarity(embeddings)
        np.fill_diagonal(sim_matrix, 0.0)
        avg_sims = sim_matrix.sum(axis=1) / (n - 1)
        diversity = 1.0 - avg_sims
        diversity = np.clip(diversity, 1e-9, None)
        probs = diversity / diversity.sum()
        raw_entropy = shannon_entropy(probs)
        max_entropy = np.log(n)
        score = float(np.clip(raw_entropy / max_entropy, 0.0, 1.0)) if max_entropy > 0 else 0.0
        uncertain = score > self.threshold

        return {
            "n_answers": n,
            "score": round(score, 4),
            "uncertain": uncertain,
            "threshold": self.threshold,
            "avg_pairwise_sim": round(float(np.mean(avg_sims)), 4),
            "raw_entropy": round(float(raw_entropy), 4),
            "max_entropy": round(float(max_entropy), 4),
            "per_answer_sims": [round(float(s), 4) for s in avg_sims],
            "verdict": "UNCERTAIN → trigger CoT" if uncertain else "CONFIDENT → pass through",
        }


if __name__ == "__main__":
    from datasets import load_dataset
    import random
    random.seed(42)

    scorer = SemanticEntropyScorer()

    print("\n[Smoke Test] Loading HaluEval samples from dataset ...")
    halueval = load_dataset("pminervini/HaluEval", "qa_samples", split="data")
    indices = random.sample(range(len(halueval)), 2)
    row_uncertain = halueval[indices[0]]

    q1 = row_uncertain["question"]
    uncertain_answers = [
        row_uncertain["hallucinated_answer"],
        row_uncertain["right_answer"],
        row_uncertain["hallucinated_answer"],
        row_uncertain["right_answer"],
        row_uncertain["hallucinated_answer"],
    ]

    print(f"\n{'='*60}")
    print(f"Test 1 — HaluEval mixed hal+right (HIGH entropy expected)")
    print(f"{'='*60}")
    print(f"  Question         : {q1}")
    print(f"  Hallucinated Ans : {row_uncertain['hallucinated_answer']}")
    print(f"  Right Answer     : {row_uncertain['right_answer']}")
    details = scorer.describe(uncertain_answers)
    for k, v in details.items():
        print(f"  {k:<22}: {v}")

    print(f"\n[Smoke Test] Loading TruthfulQA samples from dataset ...")
    tqa = load_dataset("truthful_qa", "generation", split="validation")
    tqa_row = random.choice([r for r in tqa if len(r["correct_answers"]) >= 3])
    q2 = tqa_row["question"]
    confident_answers = tqa_row["correct_answers"][:5]

    print(f"\n{'='*60}")
    print(f"Test 2 — TruthfulQA correct_answers (LOW entropy expected)")
    print(f"{'='*60}")
    print(f"  Question  : {q2}")
    for i, a in enumerate(confident_answers, 1):
        print(f"  Correct Answer {i} : {a}")
    details = scorer.describe(confident_answers)
    for k, v in details.items():
        print(f"  {k:<22}: {v}")

    print(f"\n[Smoke Test] Pulling 5 hallucinated answers across different rows ...")
    hal_rows = random.sample(range(len(halueval)), 5)
    hal_answers = [halueval[i]["hallucinated_answer"] for i in hal_rows]
    hal_questions = [halueval[i]["question"] for i in hal_rows]

    print(f"\n{'='*60}")
    print(f"Test 3 — 5 different HaluEval hallucinated answers")
    print(f"{'='*60}")
    for i, (q, a) in enumerate(zip(hal_questions, hal_answers), 1):
        print(f"  [{i}] Q: {q[:60]}...")
        print(f"       A: {a[:80]}")
    details = scorer.describe(hal_answers)
    for k, v in details.items():
        print(f"  {k:<22}: {v}")

Writing /content/drive/MyDrive/LLMwithreasoner/semantic_entropy.py


In [ ]:
%%writefile /content/drive/MyDrive/LLMwithreasoner/train_classifier.py
import os
import csv
import time
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score,
    recall_score, classification_report, confusion_matrix,
)
import numpy as np
from tqdm import tqdm

import config
from data_pipeline import get_tokenizer, get_halueval_loaders
from scratch_transformer import ScratchTransformerClassifier


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels         = labels.to(device)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss  = total_loss / len(loader)
    accuracy  = accuracy_score(all_labels, all_preds)
    f1        = f1_score(all_labels, all_preds, average="binary", zero_division=0)
    precision = precision_score(all_labels, all_preds, average="binary", zero_division=0)
    recall    = recall_score(all_labels, all_preds, average="binary", zero_division=0)

    return avg_loss, accuracy, f1, precision, recall, all_preds, all_labels


def train():
    print("\n" + "="*70)
    print("  TRAINING: Scratch Transformer Hallucination Classifier")
    print("="*70)

    device = config.DEVICE
    print(f"[Train] Using device: {device}\n")

    tokenizer = get_tokenizer()
    train_loader, val_loader = get_halueval_loaders(
        tokenizer,
        batch_size  = config.BATCH_SIZE,
        train_split = config.TRAIN_SPLIT,
        seed        = config.RANDOM_SEED,
    )

    model = ScratchTransformerClassifier().to(device)
    print(f"\n[Model] Parameters: {model.count_parameters():,}")

    criterion = nn.CrossEntropyLoss()

    optimizer = AdamW(
        model.parameters(),
        lr           = config.LEARNING_RATE,
        weight_decay = config.WEIGHT_DECAY,
    )

    total_steps = len(train_loader) * config.NUM_EPOCHS
    scheduler = OneCycleLR(
        optimizer,
        max_lr          = config.LEARNING_RATE,
        total_steps     = total_steps,
        pct_start       = 0.1,
        anneal_strategy = "cos",
    )

    best_val_f1      = 0.0
    patience_counter = 0
    patience         = 3

    csv_path = os.path.join(config.LOG_DIR, "training_log.csv")
    csv_file = open(csv_path, "w", newline="")
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow([
        "epoch", "train_loss", "val_loss",
        "train_acc", "val_acc",
        "train_f1", "val_f1",
        "val_precision", "val_recall", "lr"
    ])

    print(f"\n[Train] Starting training for {config.NUM_EPOCHS} epochs ...")
    print(f"[Train] Log → {csv_path}\n")

    for epoch in range(1, config.NUM_EPOCHS + 1):
        t_start = time.time()
        model.train()

        train_loss = 0.0
        train_preds, train_labels = [], []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{config.NUM_EPOCHS} [Train]", leave=False)
        for input_ids, attention_mask, labels in pbar:
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels         = labels.to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            train_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)
            train_preds.extend(preds.detach().cpu().numpy())
            train_labels.extend(labels.detach().cpu().numpy())
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_train_loss = train_loss / len(train_loader)
        train_acc      = accuracy_score(train_labels, train_preds)
        train_f1       = f1_score(train_labels, train_preds, average="binary", zero_division=0)

        val_loss, val_acc, val_f1, val_prec, val_rec, val_preds, val_lbls = \
            evaluate(model, val_loader, criterion, device)

        current_lr = scheduler.get_last_lr()[0]
        elapsed    = time.time() - t_start

        print(
            f"Epoch {epoch:02d}/{config.NUM_EPOCHS}  "
            f"| Train Loss: {avg_train_loss:.4f}  Acc: {train_acc:.4f}  F1: {train_f1:.4f}"
            f"  ||  Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}  F1: {val_f1:.4f}"
            f"  Prec: {val_prec:.4f}  Rec: {val_rec:.4f}"
            f"  |  LR: {current_lr:.2e}  [{elapsed:.1f}s]"
        )

        csv_writer.writerow([
            epoch, avg_train_loss, val_loss,
            train_acc, val_acc,
            train_f1, val_f1,
            val_prec, val_rec, current_lr
        ])
        csv_file.flush()

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({
                "epoch":       epoch,
                "model_state": model.state_dict(),
                "optimizer":   optimizer.state_dict(),
                "val_f1":      val_f1,
                "val_acc":     val_acc,
            }, config.CLASSIFIER_CKPT)
            print(f"  ✓  Saved best model  (val_f1={val_f1:.4f})  → {config.CLASSIFIER_CKPT}")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"  ✗  No improvement  (patience {patience_counter}/{patience})")
            if patience_counter >= patience:
                print(f"\n[Train] Early stopping triggered at epoch {epoch}.")
                break

    csv_file.close()

    print(f"\n{'='*70}")
    print("  FINAL EVALUATION (Best Checkpoint on Val Set)")
    print("="*70)

    checkpoint = torch.load(config.CLASSIFIER_CKPT, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    _, _, _, _, _, final_preds, final_labels = evaluate(model, val_loader, criterion, device)

    print("\nClassification Report:")
    print(classification_report(final_labels, final_preds, target_names=["Factual", "Hallucination"]))
    cm = confusion_matrix(final_labels, final_preds)
    print(f"Confusion Matrix:\n{cm}")
    print(f"\nBest Val F1   : {best_val_f1:.4f}")
    print(f"Checkpoint    : {config.CLASSIFIER_CKPT}")
    print(f"Training Log  : {csv_path}")

    return model


def load_trained_classifier(checkpoint_path=config.CLASSIFIER_CKPT, device=config.DEVICE):
    model = ScratchTransformerClassifier().to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    print(f"[Classifier] Loaded checkpoint from {checkpoint_path}  "
          f"(val_f1={checkpoint.get('val_f1', 0):.4f})")
    return model


if __name__ == "__main__":
    train()

Writing /content/drive/MyDrive/LLMwithreasoner/train_classifier.py


In [ ]:
%%writefile /content/drive/MyDrive/LLMwithreasoner/llama_generator.py
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import config


PASS1_SYSTEM = (
    "You are a knowledgeable and precise assistant. "
    "Answer the user's question concisely and factually."
)

PASS1_PROMPT = (
    "Question: {question}\n"
    "Answer:"
)

COT_SYSTEM = (
    "You are a careful, fact-checking assistant. "
    "Your previous answer may have been incorrect. "
    "Think step-by-step to verify the facts before giving your final answer."
)

COT_PROMPT = (
    "Question: {question}\n\n"
    "My internal supervisor flagged the previous answer as potentially false.\n"
    "Let's think step-by-step and verify the facts:\n\n"
    "Step 1: What do we know for certain about this topic?\n"
    "Step 2: What is the common misconception or myth?\n"
    "Step 3: What does evidence / logic say?\n"
    "Step 4: Final verified answer.\n\n"
    "Response:"
)

SAMPLING_PROMPT = (
    "Question: {question}\n"
    "Provide a direct answer:"
)


class LlamaGenerator:

    def __init__(self):
        print(f"\n[Llama] Loading {config.LLAMA_MODEL_ID} ...")

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        print("[Llama] Quantisation: 4-bit NF4 (bitsandbytes)")

        self.tokenizer = AutoTokenizer.from_pretrained(config.LLAMA_MODEL_ID)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(
            config.LLAMA_MODEL_ID,
            quantization_config=quant_config,
            device_map="auto",
        )
        self.model.eval()

        device_map_info = getattr(self.model, "hf_device_map", f"single device → {config.DEVICE}")
        print(f"[Llama] Ready  |  Device map: {device_map_info}")

    def _build_chat_prompt(self, system, user):
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        return prompt

    def _generate(self, prompt, max_new_tokens, temperature=0.0, do_sample=False, num_return_seq=1):
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048,
        ).to(self.model.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature if do_sample else 1.0,
                do_sample=do_sample,
                num_return_sequences=num_return_seq,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[1]
        outputs = []
        for ids in output_ids:
            new_tokens = ids[prompt_len:]
            text = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            outputs.append(text)
        return outputs

    def generate_answer(self, question):
        prompt = self._build_chat_prompt(
            system=PASS1_SYSTEM,
            user=PASS1_PROMPT.format(question=question),
        )
        answers = self._generate(
            prompt=prompt,
            max_new_tokens=config.LLAMA_MAX_NEW_TOKENS,
            temperature=0.0,
            do_sample=False,
        )
        return answers[0]

    def sample_n_answers(self, question, n=config.LLAMA_N_SAMPLES):
        prompt = self._build_chat_prompt(
            system=PASS1_SYSTEM,
            user=SAMPLING_PROMPT.format(question=question),
        )
        answers = self._generate(
            prompt=prompt,
            max_new_tokens=config.LLAMA_MAX_NEW_TOKENS,
            temperature=config.LLAMA_TEMPERATURE,
            do_sample=True,
            num_return_seq=n,
        )
        return answers

    def generate_cot_answer(self, question):
        prompt = self._build_chat_prompt(
            system=COT_SYSTEM,
            user=COT_PROMPT.format(question=question),
        )
        answers = self._generate(
            prompt=prompt,
            max_new_tokens=config.LLAMA_COT_MAX_TOKENS,
            temperature=0.0,
            do_sample=False,
        )
        return answers[0]


if __name__ == "__main__":
    import random
    from datasets import load_dataset
    random.seed(42)

    print("\n[Smoke Test] Loading TruthfulQA sample ...")
    tqa = load_dataset("truthful_qa", "generation", split="validation")
    sample = random.choice(list(tqa))
    question = sample["question"]
    best_answer = sample["best_answer"]
    correct_answers = sample["correct_answers"]

    print(f"\n{'─'*65}")
    print(f"  Question        : {question}")
    print(f"  Best Answer     : {best_answer}")
    print(f"  Correct Answers : {correct_answers[:2]}")
    print(f"{'─'*65}")

    generator = LlamaGenerator()

    print(f"\n[Pass 1 — Greedy]")
    answer_p1 = generator.generate_answer(question)
    print(f"  {answer_p1}")

    print(f"\n[Sampled Answers — N=3 for entropy]")
    sampled = generator.sample_n_answers(question, n=3)
    for i, a in enumerate(sampled, 1):
        print(f"  [{i}] {a}")

    print(f"\n[Pass 2 — CoT Re-Reasoning]")
    answer_cot = generator.generate_cot_answer(question)
    print(f"  {answer_cot}")

    print(f"\n{'─'*65}")
    print(f"  REFERENCE : {best_answer}")
    print(f"  PASS-1    : {answer_p1[:120]}")
    print(f"  CoT       : {answer_cot[:120]}")
    print(f"{'─'*65}")

Writing /content/drive/MyDrive/LLMwithreasoner/llama_generator.py


In [ ]:
import re

filepath = "/content/drive/MyDrive/LLMwithreasoner/llama_generator.py"

with open(filepath, "r") as f:
    content = f.read()

old = 'print(f"[Llama] Ready  |  Device map: {self.model.hf_device_map}")'
new = ('device_map_info = getattr(self.model, "hf_device_map", f"single device → {config.DEVICE}")\n'
       '        print(f"[Llama] Ready  |  Device map: {device_map_info}")')
content = content.replace(old, new)

content = content.replace("torch_dtype         = torch.float16,",
                           "dtype               = torch.float16,")

with open(filepath, "w") as f:
    f.write(content)

print(" llamagenerator.py patched successfully!")

 llamagenerator.py patched successfully!


In [ ]:
with open("/content/drive/MyDrive/LLMwithreasoner/llama_generator.py") as f:
    for i, line in enumerate(f, 1):
        if "device_map_info" in line or "hf_device_map" in line or "dtype" in line:
            print(f"Line {i}: {line.rstrip()}")

Line 46:             bnb_4bit_compute_dtype=torch.float16,
Line 62:         device_map_info = getattr(self.model, "hf_device_map", f"single device → {config.DEVICE}")
Line 63:         print(f"[Llama] Ready  |  Device map: {device_map_info}")


In [ ]:
filepath = "/content/drive/MyDrive/LLMwithreasoner/llama_generator.py"

with open(filepath, "r") as f:
    content = f.read()

import re
content = re.sub(r'torch_dtype\s*=\s*torch\.float16', 'dtype=torch.float16', content)

with open(filepath, "w") as f:
    f.write(content)

print("Fixed!")

Fixed!


In [ ]:
%%writefile /content/drive/MyDrive/LLMwithreasoner/eaar_pipeline.py
import torch
from transformers import AutoTokenizer
import config
from llama_generator import LlamaGenerator
from semantic_entropy import SemanticEntropyScorer
from scratch_transformer import ScratchTransformerClassifier
from train_classifier import load_trained_classifier


class EAARPipeline:

    def __init__(
        self,
        classifier_ckpt=config.CLASSIFIER_CKPT,
        entropy_threshold=config.ENTROPY_THRESHOLD,
        run_entropy=True,
    ):
        print("\n" + "="*65)
        print("  Initialising EAAR Pipeline")
        print("="*65)

        self.entropy_threshold = entropy_threshold
        self.run_entropy = run_entropy
        self.device = config.DEVICE

        print("\n[EAAR] Step 1/3 — Loading Llama-3 Generator ...")
        self.generator = LlamaGenerator()

        print("\n[EAAR] Step 2/3 — Loading Scratch Transformer Classifier ...")
        self.classifier = load_trained_classifier(classifier_ckpt, self.device)
        self.clf_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

        if run_entropy:
            print("\n[EAAR] Step 3/3 — Loading Semantic Entropy Scorer ...")
            self.entropy_scorer = SemanticEntropyScorer(threshold=entropy_threshold)
        else:
            self.entropy_scorer = None
            print("\n[EAAR] Step 3/3 — Semantic Entropy: DISABLED")

        print("\n[EAAR] ✓ Pipeline ready!\n")

    def _classify(self, question, answer):
        encoding = self.clf_tokenizer(
            question,
            answer,
            max_length=config.MAX_SEQ_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].to(self.device)
        attention_mask = encoding["attention_mask"].to(self.device)
        proba = self.classifier.predict_proba(input_ids, attention_mask)
        pred = int(torch.argmax(proba, dim=-1).item())
        hal_prob = float(proba[0, 1].item())
        return pred, hal_prob

    def _entropy(self, question):
        if self.entropy_scorer is None:
            return False, 0.0
        sampled_answers = self.generator.sample_n_answers(question, n=config.LLAMA_N_SAMPLES)
        uncertain, score = self.entropy_scorer.is_uncertain(sampled_answers)
        return uncertain, score

    def run(self, question, verbose=True):
        if verbose:
            print(f"\n{'─'*65}")
            print(f"  Question: {question}")
            print(f"{'─'*65}")

        result = {
            "question": question,
            "pass1_answer": "",
            "final_answer": "",
            "classifier_pred": -1,
            "classifier_proba": 0.0,
            "entropy_score": 0.0,
            "gate_triggered": False,
            "trigger_reason": "none",
        }

        if verbose:
            print("\n[Step 1] Llama-3 → Pass-1 Answer ...")
        pass1_answer = self.generator.generate_answer(question)
        result["pass1_answer"] = pass1_answer
        if verbose:
            print(f"  Pass-1: {pass1_answer}")

        if verbose:
            print("\n[Step 2A] Scratch Transformer Classifier ...")
        clf_pred, clf_proba = self._classify(question, pass1_answer)
        result["classifier_pred"] = clf_pred
        result["classifier_proba"] = clf_proba
        clf_label = "HALLUCINATION" if clf_pred == 1 else "FACTUAL"
        if verbose:
            print(f"  Prediction : {clf_label}  (P(hallucination)={clf_proba:.3f})")

        if verbose and self.run_entropy:
            print(f"\n[Step 2B] Semantic Entropy (sampling {config.LLAMA_N_SAMPLES} answers) ...")
        entropy_uncertain, entropy_score = self._entropy(question)
        result["entropy_score"] = entropy_score
        if verbose and self.run_entropy:
            print(f"  Entropy    : {entropy_score:.4f}  "
                  f"({'UNCERTAIN' if entropy_uncertain else 'CONFIDENT'}  "
                  f"threshold={self.entropy_threshold})")

        clf_flags = (clf_pred == 1)
        entropy_flags = entropy_uncertain

        trigger_reason = "none"
        if clf_flags and entropy_flags:
            trigger_reason = "both"
        elif clf_flags:
            trigger_reason = "classifier"
        elif entropy_flags:
            trigger_reason = "entropy"

        gate_triggered = (clf_flags or entropy_flags)
        result["gate_triggered"] = gate_triggered
        result["trigger_reason"] = trigger_reason

        if verbose:
            print(f"\n[Step 3] EAAR Gate → Triggered: {gate_triggered}  (reason: {trigger_reason})")

        if gate_triggered:
            if verbose:
                print("\n[Step 4] Triggering CoT Re-Reasoning in Llama-3 ...")
            cot_answer = self.generator.generate_cot_answer(question)
            result["final_answer"] = cot_answer
            if verbose:
                print(f"\n  CoT Answer:\n  {cot_answer}")
        else:
            result["final_answer"] = pass1_answer
            if verbose:
                print("\n[Step 4] Pass-1 answer accepted (no hallucination detected).")

        if verbose:
            print(f"\n{'─'*65}")
            print(f"  FINAL ANSWER: {result['final_answer']}")
            print(f"{'─'*65}")

        return result

    def run_batch(self, questions, verbose=False):
        results = []
        for i, q in enumerate(questions, 1):
            print(f"\n[Batch] Processing {i}/{len(questions)} ...")
            result = self.run(q, verbose=verbose)
            results.append(result)
        return results


if __name__ == "__main__":
    import random
    from datasets import load_dataset
    random.seed(42)

    print("\n[Demo] Loading TruthfulQA samples ...")
    tqa = load_dataset("truthful_qa", "generation", split="validation")
    samples = random.sample(list(tqa), 3)

    pipeline = EAARPipeline()

    for i, sample in enumerate(samples, 1):
        question = sample["question"]
        best_answer = sample["best_answer"]

        print(f"\n{'═'*65}")
        print(f"  TruthfulQA Sample {i}/3")
        print(f"  Reference Answer: {best_answer}")
        print(f"{'═'*65}")

        result = pipeline.run(question, verbose=True)

        print(f"\n  ╔══ RESULT SUMMARY ══════════════════════════════════╗")
        print(f"  ║ Gate Triggered  : {'YES' if result['gate_triggered'] else 'NO'}")
        print(f"  ║ Trigger Reason  : {result['trigger_reason']}")
        print(f"  ║ Entropy Score   : {result['entropy_score']:.4f}  (threshold={config.ENTROPY_THRESHOLD})")
        print(f"  ║ Classifier Pred : {'HALLUCINATION' if result['classifier_pred'] == 1 else 'FACTUAL'}"
              f"  (conf={result['classifier_proba']:.3f})")
        print(f"  ║ Reference Ans   : {best_answer[:55]}...")


Writing /content/drive/MyDrive/LLMwithreasoner/eaar_pipeline.py


In [ ]:
%%writefile /content/drive/MyDrive/LLMwithreasoner/evaluate.py
import os
import json
import time
import csv
import numpy as np
from tqdm import tqdm
from collections import Counter

import config
from data_pipeline import load_truthfulqa
from eaar_pipeline import EAARPipeline


def rouge_l(prediction, references):
    from rouge_score import rouge_scorer as rs
    scorer = rs.RougeScorer(["rougeL"], use_stemmer=True)
    scores = [scorer.score(ref, prediction)["rougeL"].fmeasure for ref in references]
    return max(scores) if scores else 0.0


def bert_score_batch(predictions, references):
    try:
        from bert_score import score as bert_score
        pred_list, ref_list = [], []
        for pred, refs in zip(predictions, references):
            for ref in refs:
                pred_list.append(pred)
                ref_list.append(ref)

        _, _, F = bert_score(pred_list, ref_list, lang="en", verbose=False)
        F = F.numpy()

        scores = []
        idx = 0
        for refs in references:
            n = len(refs)
            scores.append(float(F[idx:idx+n].max()))
            idx += n
        return scores

    except ImportError:
        print("[Warning] bert_score not installed. Skipping BERTScore.")
        return [0.0] * len(predictions)


def evaluate_pipeline(
    pipeline,
    max_questions=100,
    save_results=True,
    results_name="eval_results",
):
    print(f"\n{'='*65}")
    print(f"  Evaluating EAAR Pipeline on TruthfulQA  (n={max_questions})")
    print(f"{'='*65}")

    questions_data = load_truthfulqa(max_samples=max_questions)

    all_results  = []
    pass1_rougeL = []
    final_rougeL = []
    latencies    = []
    gate_reasons = []

    for item in tqdm(questions_data, desc="Running EAAR Pipeline"):
        question        = item["question"]
        correct_answers = item["correct_answers"]

        t0      = time.time()
        result  = pipeline.run(question, verbose=False)
        elapsed = time.time() - t0

        p1_score    = rouge_l(result["pass1_answer"],  correct_answers)
        final_score = rouge_l(result["final_answer"],  correct_answers)

        result["correct_answers"]  = correct_answers
        result["pass1_rougeL"]     = round(p1_score, 4)
        result["final_rougeL"]     = round(final_score, 4)
        result["latency_seconds"]  = round(elapsed, 2)
        result["improved"]         = final_score > p1_score + 0.01

        all_results.append(result)
        pass1_rougeL.append(p1_score)
        final_rougeL.append(final_score)
        latencies.append(elapsed)
        gate_reasons.append(result["trigger_reason"])

    print("\n[Evaluate] Computing BERTScore ...")
    pass1_answers = [r["pass1_answer"]   for r in all_results]
    final_answers = [r["final_answer"]   for r in all_results]
    ref_sets      = [r["correct_answers"] for r in all_results]

    pass1_bert = bert_score_batch(pass1_answers, ref_sets)
    final_bert = bert_score_batch(final_answers, ref_sets)

    for i, r in enumerate(all_results):
        r["pass1_bert"] = round(pass1_bert[i], 4)
        r["final_bert"] = round(final_bert[i], 4)

    n           = len(all_results)
    gate_counts = Counter(gate_reasons)
    n_triggered = sum(1 for r in all_results if r["gate_triggered"])
    n_improved  = sum(1 for r in all_results if r.get("improved", False))

    summary = {
        "total_questions":          n,
        "gate_triggered_count":     n_triggered,
        "gate_triggered_pct":       round(100 * n_triggered / n, 2),
        "trigger_breakdown":        dict(gate_counts),
        "pass1_rougeL_mean":        round(float(np.mean(pass1_rougeL)), 4),
        "final_rougeL_mean":        round(float(np.mean(final_rougeL)), 4),
        "rougeL_delta":             round(float(np.mean(final_rougeL)) - float(np.mean(pass1_rougeL)), 4),
        "pass1_bert_mean":          round(float(np.mean(pass1_bert)),   4),
        "final_bert_mean":          round(float(np.mean(final_bert)),   4),
        "bert_delta":               round(float(np.mean(final_bert))  - float(np.mean(pass1_bert)),  4),
        "improved_count":           n_improved,
        "improved_pct":             round(100 * n_improved / max(n_triggered, 1), 2),
        "avg_latency_seconds":      round(float(np.mean(latencies)), 2),
        "triggered_latency_mean":   round(float(np.mean([
            r["latency_seconds"] for r in all_results if r["gate_triggered"]
        ] or [0])), 2),
        "passthrough_latency_mean": round(float(np.mean([
            r["latency_seconds"] for r in all_results if not r["gate_triggered"]
        ] or [0])), 2),
    }

    print(f"\n{'='*65}")
    print("  EVALUATION SUMMARY")
    print(f"{'='*65}")
    print(f"  Questions evaluated      : {summary['total_questions']}")
    print(f"  Gate triggered           : {summary['gate_triggered_count']}  ({summary['gate_triggered_pct']}%)")
    print(f"  Trigger breakdown        : {summary['trigger_breakdown']}")
    print(f"\n  RougeL (Pass-1)          : {summary['pass1_rougeL_mean']:.4f}")
    print(f"  RougeL (Final / EAAR)    : {summary['final_rougeL_mean']:.4f}  (Δ = {summary['rougeL_delta']:+.4f})")
    print(f"\n  BERTScore (Pass-1)       : {summary['pass1_bert_mean']:.4f}")
    print(f"  BERTScore (Final / EAAR) : {summary['final_bert_mean']:.4f}  (Δ = {summary['bert_delta']:+.4f})")
    print(f"\n  Triggered & improved     : {summary['improved_count']}  ({summary['improved_pct']}% of triggered)")
    print(f"\n  Avg latency (all)        : {summary['avg_latency_seconds']}s")
    print(f"  Avg latency (triggered)  : {summary['triggered_latency_mean']}s")
    print(f"  Avg latency (pass-thru)  : {summary['passthrough_latency_mean']}s")
    print(f"{'='*65}\n")

    if save_results:
        json_path = os.path.join(config.RESULTS_DIR, f"{results_name}.json")
        with open(json_path, "w") as f:
            json.dump({"summary": summary, "results": all_results}, f, indent=2)
        print(f"[Evaluate] Saved full results → {json_path}")

        csv_path = os.path.join(config.RESULTS_DIR, f"{results_name}_summary.csv")
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=summary.keys())
            writer.writeheader()
            writer.writerow(summary)
        print(f"[Evaluate] Saved summary CSV → {csv_path}")

        per_q_path = os.path.join(config.RESULTS_DIR, f"{results_name}_per_question.csv")
        fieldnames = [
            "question", "gate_triggered", "trigger_reason",
            "classifier_pred", "classifier_proba", "entropy_score",
            "pass1_rougeL", "final_rougeL", "pass1_bert", "final_bert",
            "latency_seconds", "improved",
        ]
        with open(per_q_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(all_results)
        print(f"[Evaluate] Saved per-question CSV → {per_q_path}")

    return summary, all_results


if __name__ == "__main__":
    pipeline = EAARPipeline()
    summary, results = evaluate_pipeline(pipeline, max_questions=100)

Writing /content/drive/MyDrive/LLMwithreasoner/evaluate.py


In [ ]:
# CELL 1 — Fix column names in data_pipeline.py
filepath = "/content/drive/MyDrive/LLMwithreasoner/data_pipeline.py"

with open(filepath, "r") as f:
    content = f.read()

content = content.replace('row.get("right_answer", "")',        'row.get("answer", "")')
content = content.replace('row.get("hallucinated_answer", "")', 'row.get("hallucination", "")')

with open(filepath, "w") as f:
    f.write(content)

print(" Column names fixed!")

 Column names fixed!


In [ ]:
# CELL 2 — Add class weights to train_classifier.py
filepath = "/content/drive/MyDrive/LLMwithreasoner/train_classifier.py"

with open(filepath, "r") as f:
    content = f.read()

old = "    criterion = nn.CrossEntropyLoss()"

new = """    all_labels_list = []
    for _, _, lbl in train_loader:
        all_labels_list.extend(lbl.numpy().tolist())
    n_total  = len(all_labels_list)
    n_class0 = all_labels_list.count(0)
    n_class1 = all_labels_list.count(1)
    w0       = n_total / (2 * n_class0) if n_class0 > 0 else 1.0
    w1       = n_total / (2 * n_class1) if n_class1 > 0 else 1.0
    weights  = torch.tensor([w0, w1], dtype=torch.float).to(device)
    print(f"[Train] Class weights → Factual={w0:.3f}  Hallucination={w1:.3f}")
    criterion = nn.CrossEntropyLoss(weight=weights)"""

content = content.replace(old, new)

with open(filepath, "w") as f:
    f.write(content)

print(" Class weights added!")

 Class weights added!


In [ ]:
# CELL 3 — Fix load_trained_classifier to infer arch from checkpoint weights
filepath = "/content/drive/MyDrive/LLMwithreasoner/train_classifier.py"

with open(filepath, "r") as f:
    content = f.read()

import re

new_load = '''def load_trained_classifier(checkpoint_path=config.CLASSIFIER_CKPT, device=config.DEVICE):
    ckpt      = torch.load(checkpoint_path, map_location=device)
    state     = ckpt["model_state"]
    d_model   = state["token_embedding.weight"].shape[1]
    n_layers  = sum(1 for k in state if k.startswith("layers.") and k.endswith("norm1.weight"))
    vocab_size = state["token_embedding.weight"].shape[0]
    model = ScratchTransformerClassifier(
        vocab_size  = vocab_size,
        d_model     = d_model,
        n_heads     = config.N_HEADS,
        n_layers    = n_layers,
        d_ff        = d_model * 2,
        max_len     = config.MAX_SEQ_LEN,
        dropout     = config.DROPOUT,
        num_classes = config.NUM_CLASSES,
    ).to(device)
    model.load_state_dict(state)
    model.eval()
    print(f"[Classifier] Loaded → d_model={d_model}  n_layers={n_layers}  val_f1={ckpt.get(\'val_f1\', 0):.4f}")
    return model'''

content = re.sub(
    r'def load_trained_classifier.*?return model',
    new_load,
    content,
    flags=re.DOTALL
)

with open(filepath, "w") as f:
    f.write(content)

print(" load_trained_classifier fixed!")

 load_trained_classifier fixed!


In [ ]:
# CELL 4 — Delete any old checkpoint
import os
ckpt = "/content/drive/MyDrive/LLMwithreasoner/checkpoints/scratch_transformer_best.pt"
if os.path.exists(ckpt):
    os.remove(ckpt)
    print(" Old checkpoint deleted")
else:
    print(" No old checkpoint — clean start")

 No old checkpoint — clean start


In [ ]:
# CELL 5 — Verify all 3 patches applied correctly
print("Checking patches...\n")

with open("/content/drive/MyDrive/LLMwithreasoner/data_pipeline.py") as f:
    dp = f.read()
print("data_pipeline.py:")
print("  Column 'answer'      :", 'row.get("answer"'       in dp)
print("  Column 'hallucination':", 'row.get("hallucination"' in dp)

with open("/content/drive/MyDrive/LLMwithreasoner/train_classifier.py") as f:
    tc = f.read()
print("\ntrain_classifier.py:")
print("  Class weights        :", "n_class0" in tc)
print("  Infer arch from ckpt :", 'd_model   = state["token_embedding' in tc)

Checking patches...

data_pipeline.py:
  Column 'answer'      : True
  Column 'hallucination': True

train_classifier.py:
  Class weights        : True
  Infer arch from ckpt : True


In [ ]:
# CELL 6 — Train (only run after Cell 5 shows all True)
%cd /content/drive/MyDrive/LLMwithreasoner
!python main.py --mode train

/content/drive/MyDrive/LLMwithreasoner
[Config] Device       : cuda
[Config] Llama Model  : meta-llama/Meta-Llama-3-8B-Instruct
[Config] Classifier   : Scratch Transformer  (d=128, heads=4, layers=3)
[Config] Entropy Thr. : 1

    ╔══════════════════════════════════════════════════════════════╗
    ║                                                              ║
    ║      EAAR — Error-Aware Adaptive Re-Reasoning System        ║
    ║                                                              ║
    ║   Generator  : Llama-3-8B-Instruct                          ║
    ║   Supervisor : Scratch Transformer (trained on HaluEval)    ║
    ║   Gate Logic : (Classifier == Hallucination) OR             ║
    ║                (Semantic Entropy > 0.6)                      ║
    ║             → Triggers CoT Re-Reasoning                     ║
    ║                                                              ║
    ╚══════════════════════════════════════════════════════════════╝
    

 Mode: TRAIN


In [ ]:
%cd /content/drive/MyDrive/LLMwithreasoner
!python main.py --mode demo

/content/drive/MyDrive/LLMwithreasoner
[Config] Device       : cuda
[Config] Llama Model  : meta-llama/Meta-Llama-3-8B-Instruct
[Config] Classifier   : Scratch Transformer  (d=128, heads=4, layers=3)
[Config] Entropy Thr. : 1

    ╔══════════════════════════════════════════════════════════════╗
    ║                                                              ║
    ║      EAAR — Error-Aware Adaptive Re-Reasoning System        ║
    ║                                                              ║
    ║   Generator  : Llama-3-8B-Instruct                          ║
    ║   Supervisor : Scratch Transformer (trained on HaluEval)    ║
    ║   Gate Logic : (Classifier == Hallucination) OR             ║
    ║                (Semantic Entropy > 0.6)                      ║
    ║             → Triggers CoT Re-Reasoning                     ║
    ║                                                              ║
    ╚══════════════════════════════════════════════════════════════╝
    
Traceback (mos

In [ ]:
# CELL 2 — Fix both root causes in config.py
filepath = "/content/drive/MyDrive/LLMwithreasoner/config.py"

import re
with open(filepath, "r") as f:
    content = f.read()

# Fix 1: Lower temperature so sampled answers are less randomly diverse
content = re.sub(r'LLAMA_TEMPERATURE\s*=\s*[\d.]+',  'LLAMA_TEMPERATURE     = 0.3', content)

# Fix 2: Raise entropy threshold — 0.6 is too low for LLM outputs
content = re.sub(r'ENTROPY_THRESHOLD\s*=\s*[\d.]+',  'ENTROPY_THRESHOLD     = 1', content)

# Fix 3: Reduce N samples from 5 to 3 — fewer samples = less noise
content = re.sub(r'LLAMA_N_SAMPLES\s*=\s*\d+',       'LLAMA_N_SAMPLES       = 3', content)

with open(filepath, "w") as f:
    f.write(content)

print(" config.py fixed!")

# Verify
with open(filepath, "r") as f:
    for line in f:
        if any(x in line for x in ["LLAMA_TEMPERATURE", "ENTROPY_THRESHOLD", "LLAMA_N_SAMPLES"]):
            print(f"  {line.rstrip()}")

 config.py fixed!
  LLAMA_TEMPERATURE     = 0.3
  LLAMA_N_SAMPLES       = 3
  ENTROPY_THRESHOLD     = 1
  print(f"[Config] Entropy Thr. : {ENTROPY_THRESHOLD}")


In [ ]:
%%writefile /content/drive/MyDrive/LLMwithreasoner/main.py
import argparse
import sys
import config


def run_train():
    from train_classifier import train
    print("\n Mode: TRAIN\n")
    train()
    print(f"\n Training complete. Checkpoint saved → {config.CLASSIFIER_CKPT}")


def run_evaluate(num_questions=100):
    from eaar_pipeline import EAARPipeline
    from evaluate import evaluate_pipeline
    print(f"\n Mode: EVALUATE  (n={num_questions})\n")
    pipeline = EAARPipeline()
    summary, _ = evaluate_pipeline(pipeline, max_questions=num_questions)
    print_summary_table(summary)


def run_demo(question=None):
    import random
    from datasets import load_dataset
    from eaar_pipeline import EAARPipeline

    print("\n Mode: DEMO")
    pipeline = EAARPipeline()

    if question:
        samples = [{"question": question, "best_answer": "N/A"}]
    else:
        print("\n[Demo] Loading TruthfulQA samples from dataset ...")
        random.seed(config.RANDOM_SEED)
        tqa = load_dataset("truthful_qa", "generation", split="validation")
        samples = random.sample(list(tqa), 3)

    for i, sample in enumerate(samples, 1):
        q           = sample["question"]
        best_answer = sample.get("best_answer", "N/A")

        print(f"\n{'═'*65}")
        print(f"  Demo Question {i}/{len(samples)}")
        print(f"  Reference Answer: {best_answer}")
        print(f"{'═'*65}")

        result = pipeline.run(q, verbose=True)

        print(f"\n  ╔══ RESULT SUMMARY ══════════════════════════════════╗")
        print(f"  ║ Gate Triggered  : {'YES' if result['gate_triggered'] else 'NO'}")
        print(f"  ║ Trigger Reason  : {result['trigger_reason']}")
        print(f"  ║ Entropy Score   : {result['entropy_score']:.4f}  (threshold={config.ENTROPY_THRESHOLD})")
        print(f"  ║ Classifier Pred : {'HALLUCINATION' if result['classifier_pred'] == 1 else 'FACTUAL'}"
              f"  (conf={result['classifier_proba']:.3f})")
        print(f"  ║ Reference Ans   : {best_answer[:55]}...")
        print(f"  ╚════════════════════════════════════════════════════╝")

        if i < len(samples) and not question:
            input("\n  [Press Enter for next question]")


def print_summary_table(summary):
    print(f"\n{'═'*65}")
    print("  EVALUATION RESULTS")
    print(f"{'═'*65}")
    rows = [
        ("Questions Evaluated",         summary["total_questions"]),
        ("Gate Triggered",              f"{summary['gate_triggered_count']} ({summary['gate_triggered_pct']}%)"),
        ("Trigger Breakdown",           str(summary["trigger_breakdown"])),
        ("",                            ""),
        ("RougeL — Pass 1",             f"{summary['pass1_rougeL_mean']:.4f}"),
        ("RougeL — EAAR Final",         f"{summary['final_rougeL_mean']:.4f}  (Δ {summary['rougeL_delta']:+.4f})"),
        ("",                            ""),
        ("BERTScore — Pass 1",          f"{summary['pass1_bert_mean']:.4f}"),
        ("BERTScore — EAAR Final",      f"{summary['final_bert_mean']:.4f}  (Δ {summary['bert_delta']:+.4f})"),
        ("",                            ""),
        ("Triggered & Improved",        f"{summary['improved_count']} ({summary['improved_pct']}% of triggered)"),
        ("Avg Latency (all)",           f"{summary['avg_latency_seconds']}s"),
        ("Avg Latency (triggered)",     f"{summary['triggered_latency_mean']}s"),
        ("Avg Latency (pass-through)",  f"{summary['passthrough_latency_mean']}s"),
    ]
    for label, value in rows:
        if label:
            print(f"  {label:<30}: {value}")
        else:
            print()
    print(f"{'═'*65}\n")


def print_banner():
    banner = r"""
    ╔══════════════════════════════════════════════════════════════╗
    ║                                                              ║
    ║      EAAR — Error-Aware Adaptive Re-Reasoning System        ║
    ║                                                              ║
    ║   Generator  : Llama-3-8B-Instruct                          ║
    ║   Supervisor : Scratch Transformer (trained on HaluEval)    ║
    ║   Gate Logic : (Classifier == Hallucination) OR             ║
    ║                (Semantic Entropy > 0.6)                      ║
    ║             → Triggers CoT Re-Reasoning                     ║
    ║                                                              ║
    ╚══════════════════════════════════════════════════════════════╝
    """
    print(banner)


def parse_args():
    parser = argparse.ArgumentParser(
        description="EAAR — Error-Aware Adaptive Re-Reasoning for Hallucination Detection",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--mode", type=str, choices=["train", "evaluate", "demo"], default="demo")
    parser.add_argument("--num_questions", type=int, default=100)
    parser.add_argument("--question", type=str, default=None)
    return parser.parse_args()


if __name__ == "__main__":
    print_banner()
    args = parse_args()

    if args.mode == "train":
        run_train()
    elif args.mode == "evaluate":
        run_evaluate(num_questions=args.num_questions)
    elif args.mode == "demo":
        run_demo(question=args.question)
    else:
        print(f"Unknown mode: {args.mode}")
        sys.exit(1)

Overwriting /content/drive/MyDrive/LLMwithreasoner/main.py


In [ ]:
!python /content/drive/MyDrive/LLMwithreasoner/main.py --mode demo

[Config] Device       : cuda
[Config] Llama Model  : meta-llama/Meta-Llama-3-8B-Instruct
[Config] Classifier   : Scratch Transformer  (d=128, heads=4, layers=3)
[Config] Entropy Thr. : 1

    ╔══════════════════════════════════════════════════════════════╗
    ║                                                              ║
    ║      EAAR — Error-Aware Adaptive Re-Reasoning System        ║
    ║                                                              ║
    ║   Generator  : Llama-3-8B-Instruct                          ║
    ║   Supervisor : Scratch Transformer (trained on HaluEval)    ║
    ║   Gate Logic : (Classifier == Hallucination) OR             ║
    ║                (Semantic Entropy > 0.6)                      ║
    ║             → Triggers CoT Re-Reasoning                     ║
    ║                                                              ║
    ╚══════════════════════════════════════════════════════════════╝
    

 Mode: DEMO

  Initialising EAAR Pipeline

[EAAR] St

In [ ]:
%%writefile /content/drive/MyDrive/LLMwithreasoner/metrics.py

import random
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as shannon_entropy
from rouge_score import rouge_scorer
import config
from eaar_pipeline import EAARPipeline


REPHRASE_TEMPLATES = [
    "{q}",
    "Can you tell me: {q}",
    "Please answer this question: {q}",
    "I would like to know: {q}",
    "Could you explain: {q}",
]


def get_sbert():
    return SentenceTransformer(config.SBERT_MODEL_ID)

def get_rouge():
    return rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def best_rouge(scorer, prediction, references):
    return max(scorer.score(ref, prediction)["rougeL"].fmeasure for ref in references)

def section(title):
    print(f"\n{'═'*65}")
    print(f"  {title}")
    print(f"{'═'*65}")

def divider():
    print(f"  {'─'*63}")


def metric_accuracy(results):
    section("METRIC 1 — ACCURACY")
    scorer  = get_rouge()
    correct = 0
    scores  = []

    for r in results:
        refs = r.get("correct_answers", [r.get("best_answer", "")])
        if isinstance(refs, str):
            refs = [refs]
        score = best_rouge(scorer, r["final_answer"], refs)
        scores.append(score)
        if score >= 0.3:
            correct += 1

    accuracy = correct / len(results) if results else 0.0

    print(f"\n  Total Questions  : {len(results)}")
    print(f"  Correct (>=0.3)  : {correct}")
    print(f"  Accuracy         : {accuracy*100:.2f}%")
    print(f"  Avg RougeL Score : {np.mean(scores):.4f}")
    print(f"\n  Per-Question:")
    divider()
    for i, (r, score) in enumerate(zip(results, scores), 1):
        verdict = "CORRECT" if score >= 0.3 else "WRONG  "
        print(f"  [{i:02d}] {verdict}  RougeL={score:.4f}  {r['question'][:50]}...")
    divider()
    verdict = "GOOD" if accuracy > 0.6 else "MODERATE" if accuracy > 0.4 else "LOW"
    print(f"\n  Verdict: {verdict} ACCURACY")

    return {
        "accuracy":      round(accuracy, 4),
        "correct_count": correct,
        "avg_rougeL":    round(float(np.mean(scores)), 4),
    }


def metric_hallucination_rate(results):
    section("METRIC 2 — HALLUCINATION RATE")
    scorer = get_rouge()
    total  = len(results)

    pass1_correct = 0
    final_correct = 0
    per_q         = []

    for r in results:
        refs = r.get("correct_answers", [r.get("best_answer", "")])
        if isinstance(refs, str):
            refs = [refs]
        p1_score = best_rouge(scorer, r["pass1_answer"], refs)
        fn_score = best_rouge(scorer, r["final_answer"],  refs)
        if p1_score >= 0.3:
            pass1_correct += 1
        if fn_score >= 0.3:
            final_correct += 1
        per_q.append({
            "question":    r["question"],
            "p1_score":    round(p1_score, 4),
            "final_score": round(fn_score,  4),
            "gate":        r["gate_triggered"],
        })

    hal_pass1 = 1.0 - (pass1_correct / total) if total else 0.0
    hal_final = 1.0 - (final_correct / total)  if total else 0.0
    reduction = hal_pass1 - hal_final
    flagged   = sum(1 for r in results if r["gate_triggered"])
    clf_flags = sum(1 for r in results if r["classifier_pred"] == 1)
    ent_flags = sum(1 for r in results if r["entropy_score"] > config.ENTROPY_THRESHOLD)

    print(f"\n  Total Questions          : {total}")
    print(f"  Pass-1 Correct           : {pass1_correct}/{total}")
    print(f"  Final  Correct           : {final_correct}/{total}")
    print(f"\n  Pass-1 Hallucination Rate: {hal_pass1*100:.2f}%")
    print(f"  Final  Hallucination Rate: {hal_final*100:.2f}%")
    print(f"  Hallucination Reduction  : {reduction*100:+.2f}%")
    print(f"\n  Flagged by Classifier    : {clf_flags}")
    print(f"  Flagged by Entropy       : {ent_flags}")
    print(f"  Flagged by Gate (Either) : {flagged}  ({flagged/total*100:.1f}%)")
    print(f"\n  Per-Question:")
    divider()
    print(f"  {'Question':<45} {'Pass1':>6}  {'Final':>6}  Gate")
    divider()
    for q in per_q:
        p1v  = "GOOD" if q["p1_score"]    >= 0.3 else "HAL "
        fnv  = "GOOD" if q["final_score"] >= 0.3 else "HAL "
        gate = "TRIGGERED" if q["gate"] else "         "
        print(f"  {q['question'][:45]:<45} {p1v}={q['p1_score']:.3f}  {fnv}={q['final_score']:.3f}  {gate}")
    divider()
    verdict = (
        "EAAR significantly reduced hallucinations"
        if reduction > 0.1 else
        "EAAR had minor effect"
        if reduction > 0 else
        "EAAR did not reduce hallucinations"
    )
    print(f"\n  Verdict: {verdict}")

    return {
        "pass1_hallucination_rate": round(hal_pass1, 4),
        "final_hallucination_rate": round(hal_final, 4),
        "hallucination_reduction":  round(reduction, 4),
        "gate_trigger_rate":        round(flagged / total, 4),
    }


def metric_consistency(generator, questions, n_repeats=3):
    section("METRIC 3 — CONSISTENCY SCORE")
    print(f"  (Asking each question {n_repeats}x and comparing answers)\n")
    sbert  = get_sbert()
    scores = []

    for i, question in enumerate(questions, 1):
        print(f"  [{i}/{len(questions)}] {question[:60]}...")
        answers = [generator.generate_answer(question) for _ in range(n_repeats)]
        embeddings = sbert.encode(answers, convert_to_numpy=True, show_progress_bar=False)
        sim_matrix = cosine_similarity(embeddings)
        np.fill_diagonal(sim_matrix, 0.0)
        n       = len(answers)
        avg_sim = sim_matrix.sum() / (n * (n - 1))
        scores.append((question, float(avg_sim), answers))

    avg_score = float(np.mean([s[1] for s in scores]))

    divider()
    for i, (q, score, answers) in enumerate(scores, 1):
        verdict = "HIGH" if score > 0.8 else "MEDIUM" if score > 0.5 else "LOW"
        print(f"\n  [{i:02d}] Consistency: {score:.4f}  [{verdict}]")
        print(f"        Q: {q[:65]}")
        for j, ans in enumerate(answers, 1):
            print(f"        Answer {j}: {ans[:75]}...")
    divider()
    print(f"\n  Avg Consistency Score : {avg_score:.4f}")
    print(f"  Min                   : {min(s[1] for s in scores):.4f}")
    print(f"  Max                   : {max(s[1] for s in scores):.4f}")
    verdict = (
        "HIGH — model is consistent and trustworthy"
        if avg_score > 0.8 else
        "MEDIUM — some variation in answers"
        if avg_score > 0.5 else
        "LOW — model is inconsistent, hallucination risk is high"
    )
    print(f"  Verdict               : {verdict}")

    return {
        "avg_consistency_score": round(avg_score, 4),
        "min_consistency_score": round(min(s[1] for s in scores), 4),
        "max_consistency_score": round(max(s[1] for s in scores), 4),
    }


def metric_prompt_sensitivity(generator, questions):
    section("METRIC 4 — PROMPT SENSITIVITY")
    print(f"  (Testing {len(REPHRASE_TEMPLATES)} phrasings of each question)\n")
    sbert  = get_sbert()
    scores = []

    for i, question in enumerate(questions, 1):
        print(f"  [{i}/{len(questions)}] {question[:60]}...")
        rephrased  = [t.format(q=question) for t in REPHRASE_TEMPLATES]
        answers    = [generator.generate_answer(q) for q in rephrased]
        embeddings = sbert.encode(answers, convert_to_numpy=True, show_progress_bar=False)
        sim_matrix = cosine_similarity(embeddings)
        np.fill_diagonal(sim_matrix, 0.0)
        n           = len(answers)
        avg_sim     = sim_matrix.sum() / (n * (n - 1))
        sensitivity = 1.0 - avg_sim
        scores.append((question, float(sensitivity), rephrased, answers))

    avg_score = float(np.mean([s[1] for s in scores]))

    divider()
    for i, (q, score, rephrased, answers) in enumerate(scores, 1):
        verdict = "LOW" if score < 0.2 else "MEDIUM" if score < 0.5 else "HIGH"
        print(f"\n  [{i:02d}] Sensitivity: {score:.4f}  [{verdict}]")
        print(f"        Original Q : {q[:65]}")
        for phrasing, answer in zip(rephrased[1:], answers[1:]):
            print(f"        Rephrase   : {phrasing[:65]}")
            print(f"        Answer     : {answer[:70]}...")
    divider()
    print(f"\n  Avg Sensitivity Score : {avg_score:.4f}")
    print(f"  Min                   : {min(s[1] for s in scores):.4f}")
    print(f"  Max                   : {max(s[1] for s in scores):.4f}")
    verdict = (
        "LOW — model is robust to prompt rephrasing (GOOD)"
        if avg_score < 0.2 else
        "MEDIUM — moderate sensitivity to phrasing"
        if avg_score < 0.5 else
        "HIGH — model is fragile, changes answer with phrasing (BAD)"
    )
    print(f"  Verdict               : {verdict}")

    return {
        "avg_prompt_sensitivity": round(avg_score, 4),
        "min_prompt_sensitivity": round(min(s[1] for s in scores), 4),
        "max_prompt_sensitivity": round(max(s[1] for s in scores), 4),
    }


def metric_semantic_entropy(generator, questions, n_samples=5):
    section("METRIC 5 — SEMANTIC ENTROPY")
    print(f"  (Sampling {n_samples} answers per question and measuring diversity)\n")
    sbert   = get_sbert()
    results = []

    for i, question in enumerate(questions, 1):
        print(f"  [{i}/{len(questions)}] {question[:60]}...")
        answers    = generator.sample_n_answers(question, n=n_samples)
        embeddings = sbert.encode(answers, convert_to_numpy=True, show_progress_bar=False)
        sim_matrix = cosine_similarity(embeddings)
        sim_matrix = np.clip(sim_matrix, 0.0, 1.0)
        np.fill_diagonal(sim_matrix, 0.0)
        n         = len(answers)
        avg_sims  = sim_matrix.sum(axis=1) / (n - 1)
        diversity = np.clip(1.0 - avg_sims, 1e-9, None)
        probs     = diversity / diversity.sum()
        raw_ent   = shannon_entropy(probs)
        max_ent   = np.log(n)
        score     = float(np.clip(raw_ent / max_ent, 0.0, 1.0)) if max_ent > 0 else 0.0
        uncertain = score > config.ENTROPY_THRESHOLD
        results.append({"question": question, "entropy": round(score, 4), "uncertain": uncertain, "answers": answers})

    scores    = [r["entropy"] for r in results]
    uncertain = [r for r in results if r["uncertain"]]

    divider()
    for i, item in enumerate(results, 1):
        verdict = "UNCERTAIN -> CoT triggered" if item["uncertain"] else "CONFIDENT -> pass through"
        print(f"\n  [{i:02d}] Entropy: {item['entropy']:.4f}  [{verdict}]")
        print(f"        Q: {item['question'][:65]}")
        for j, ans in enumerate(item["answers"], 1):
            print(f"        Sample {j}: {ans[:70]}...")
    divider()
    print(f"\n  Avg Entropy          : {np.mean(scores):.4f}")
    print(f"  Std Entropy          : {np.std(scores):.4f}")
    print(f"  Min Entropy          : {np.min(scores):.4f}")
    print(f"  Max Entropy          : {np.max(scores):.4f}")
    print(f"  Threshold            : {config.ENTROPY_THRESHOLD}")
    print(f"  Uncertain Count      : {len(uncertain)}/{len(results)}  ({len(uncertain)/len(results)*100:.1f}%)")
    verdict = (
        "GOOD — most answers have low semantic entropy"
        if np.mean(scores) < 0.4 else
        "MODERATE — some uncertainty in generated answers"
        if np.mean(scores) < 0.6 else
        "HIGH — model is frequently uncertain"
    )
    print(f"  Verdict              : {verdict}")

    return {
        "avg_entropy":     round(float(np.mean(scores)), 4),
        "std_entropy":     round(float(np.std(scores)),  4),
        "uncertain_count": len(uncertain),
        "uncertain_pct":   round(len(uncertain) / len(results) * 100, 2),
    }

if __name__ == "__main__":
    random.seed(config.RANDOM_SEED)

    print("\n" + ""*65)
    print("  EAAR — COMPREHENSIVE METRICS EVALUATION")
    print("  All 5 metrics printed separately one by one")
    print(""*65)

    print("\n[Setup] Loading TruthfulQA from dataset ...")
    tqa         = load_dataset("truthful_qa", "generation", split="validation")
    all_samples = random.sample(list(tqa), 30)
    questions   = [s["question"] for s in all_samples]

    print("[Setup] Initialising EAAR Pipeline ...")
    pipeline = EAARPipeline()

    print("\n[Setup] Running EAAR pipeline on 30 questions ...")
    pipeline_results = []
    for i, sample in enumerate(all_samples, 1):
        print(f"  [{i}/30] {sample['question'][:60]}...")
        result = pipeline.run(sample["question"], verbose=False)
        result["correct_answers"] = sample["correct_answers"]
        result["best_answer"]     = sample["best_answer"]
        pipeline_results.append(result)

    print("\n\n" + ""*65)
    print("  RESULTS — EACH METRIC SHOWN SEPARATELY BELOW")
    print(""*65)

    acc_result  = metric_accuracy(pipeline_results)

    input("\n  [Press Enter to continue to Metric 2]")
    hal_result  = metric_hallucination_rate(pipeline_results)

    input("\n  [Press Enter to continue to Metric 3]")
    cons_result = metric_consistency(pipeline.generator, questions[:10], n_repeats=3)

    input("\n  [Press Enter to continue to Metric 4]")
    sens_result = metric_prompt_sensitivity(pipeline.generator, questions[:10])

    input("\n  [Press Enter to continue to Metric 5]")
    ent_result  = metric_semantic_entropy(pipeline.generator, questions[:10], n_samples=config.LLAMA_N_SAMPLES)

    print(f"\n\n{''*65}")
    print(f"  FINAL COMBINED SUMMARY")
    print(f"{''*65}")
    print(f"\n  1. ACCURACY")
    print(f"     Accuracy             : {acc_result['accuracy']*100:.2f}%")
    print(f"     Avg RougeL           : {acc_result['avg_rougeL']:.4f}")
    print(f"\n  2. HALLUCINATION RATE")
    print(f"     Before EAAR (Pass-1) : {hal_result['pass1_hallucination_rate']*100:.2f}%")
    print(f"     After  EAAR (Final)  : {hal_result['final_hallucination_rate']*100:.2f}%")
    print(f"     Reduction            : {hal_result['hallucination_reduction']*100:+.2f}%")
    print(f"\n  3. CONSISTENCY SCORE")
    print(f"     Avg Score            : {cons_result['avg_consistency_score']:.4f}")
    print(f"\n  4. PROMPT SENSITIVITY")
    print(f"     Avg Score            : {sens_result['avg_prompt_sensitivity']:.4f}")
    print(f"\n  5. SEMANTIC ENTROPY")
    print(f"     Avg Entropy          : {ent_result['avg_entropy']:.4f}")
    print(f"     Uncertain Samples    : {ent_result['uncertain_count']}  ({ent_result['uncertain_pct']}%)")
    print(f"\n{''*65}")

Writing /content/drive/MyDrive/LLMwithreasoner/metrics.py


In [ ]:
%cd /content/drive/MyDrive/LLMwithreasoner
!python metrics.py

/content/drive/MyDrive/LLMwithreasoner
[Config] Device       : cuda
[Config] Llama Model  : meta-llama/Meta-Llama-3-8B-Instruct
[Config] Classifier   : Scratch Transformer  (d=128, heads=4, layers=3)
[Config] Entropy Thr. : 1


  EAAR — COMPREHENSIVE METRICS EVALUATION
  All 5 metrics printed separately one by one


[Setup] Loading TruthfulQA from dataset ...
[Setup] Initialising EAAR Pipeline ...

  Initialising EAAR Pipeline

[EAAR] Step 1/3 — Loading Llama-3 Generator ...

[Llama] Loading meta-llama/Meta-Llama-3-8B-Instruct ...
[Llama] Quantisation: 4-bit NF4 (bitsandbytes)
Loading weights: 100% 291/291 [01:06<00:00,  4.38it/s, Materializing param=model.norm.weight]
[Llama] Ready  |  Device map: single device → cuda

[EAAR] Step 2/3 — Loading Scratch Transformer Classifier ...
[Classifier] Loaded → d_model=128  n_layers=3  val_f1=0.9821

[EAAR] Step 3/3 — Loading Semantic Entropy Scorer ...
[SemanticEntropy] Loading sentence encoder: all-MiniLM-L6-v2 ...
Loading weights: 100% 103/10

In [ ]:
%cd /content
!rm -rf Hallucination-detection-and-mitigation

!git clone https://github.com/constsubhampaul/Hallucination-detection-and-mitigation.git

from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/Colab Notebooks/LLMwithreasoner.ipynb" /content/Hallucination-detection-and-mitigation/

%cd /content/Hallucination-detection-and-mitigation

!git config --global user.name "constsubhampaul"
!git config --global user.email "subhampaul620@gmail.com"

!git add .
!git commit -m "Error fixed"

import getpass
token = getpass.getpass("Enter GitHub Token: ")

!git push https://{token}@github.com/constsubhampaul/Hallucination-detection-and-mitigation.git main

/content
Cloning into 'Hallucination-detection-and-mitigation'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 17.04 KiB | 623.00 KiB/s, done.
